In [ ]:
from pathlib import Path
import json

import pandas as pd
from pymongo import MongoClient
from pymongo.errors import OperationFailure, ServerSelectionTimeoutError

In [ ]:
MONGO_CONFIG = {
    "user": "labuser",
    "password": "labpass",
    "host": "localhost",
    "port": 27017,
    # "database": "university_mongo"
}

RAW_DIR = '../raw/clean'
# COLLECTION_NAME = 'diabetes'

In [ ]:
class MongoDatabase:
    """Class responsible for MongoDB connection management."""

    def __init__(self, db_config):
        # store configuration fields such as host, port, user, password, database
        self.client = None
        self.db = None
        self.config = db_config

    def connect(self):
        if self.client is not None:
            return self.db
        
        try:
            # create a MongoClient only if it does not exist yet
            # build the URI with authSource=admin
            # set a small serverSelectionTimeoutMS for friendlier failures
            # select the target database
            # test the connection with ping

            uri = (f"mongodb://{self.config['user']}:{self.config['password']}@{self.config['host']}:{self.config['port']}/?"
                "authSource=admin")
            self.client = MongoClient(uri, serverSelectionTimeoutMS=1000)
            self.db = self.client[self.config['database']]
            self.client.admin.command('ping')
            return self.db
        
            # if authentication fails, raise a clear message mentioning
            # prj/docker-compose.yml and the docker exec mongosh command

        except OperationFailure as e:
            self.client = None
            raise OperationFailure(
                "MongoDB authentication failed.\n"
                "Check credentials in Final-Project/docker-compose.yml.\n"
                "You can test manually with:\n"
                "docker exec -it lab-mongo mongosh -u labuser -p labpass --authenticationDatabase admin", e)
            
        except ServerSelectionTimeoutError as e:
            self.client = None
            raise ServerSelectionTimeoutError(
                "Could not connect to MongoDB server. "
                "Check if the container/service is running and accessible.", e)

    def is_connected(self):
        # return True if the client is available, otherwise False
        return self.client is not None

    def get_collection(self, collection_name):
        # return the collection object from the selected database
        if self.db is None:
            raise Exception("Database not connected. Call connect() first.")
        return self.db[collection_name]

    def close(self):
        # close the client and reset internal references
        if self.client is not None:
            self.client.close()
        self.client = None
        self.db = None

In [ ]:
class MongoExecutor:
    """Class responsible for CRUD operations on a MongoDB collection."""

    def __init__(self, database, collection_name):
        # keep a reference to the database object
        # obtain the collection from the database object
        self.db = database
        self.collection = self.db.get_collection(collection_name)

    def select(self, filter_query=None, projection=None, limit=10):
        # implement a simple find() wrapper
        cursor = self.collection.find(filter_query or {}, projection).limit(limit)
        return list(cursor)
        
    def insert_one(self, document):
        # insert a single document
        return self.collection.insert_one(document)

    def insert_many(self, documents):
        # insert many documents for the ingestion step
        return self.collection.insert_many(documents)

    def update_one(self, filter_query, update_query):
        # update a single document
        return self.collection.update_one(filter_query, update_query)

    def delete_one(self, filter_query):
        # delete a single document
        return self.collection.delete_one(filter_query)
    
    def delete_many(self, filter_query=None):
        # delete many documents for ingestion step (avoid repeating documents)
        return self.collection.delete_many(filter_query or {})

    def count_documents(self, filter_query=None):
        # count documents, defaulting to all documents
        return self.collection.count_documents(filter_query or {})


In [ ]:
db = MongoDatabase(MONGO_CONFIG)
db.connect()

collection = db.get_collection(COLLECTION_NAME)
executor = MongoExecutor(db, COLLECTION_NAME)

db.client.admin.command('ping')
print("Successfully connected to MongoDB!")
print("Document count:", collection.count_documents({}))